In [0]:
from pyspark.sql.functions import (
    col, lit, explode, trim, initcap, split, element_at, coalesce,
    to_timestamp, to_date, round as spark_round,
)

bronze = spark.table("workspace.coffee.bronze_online_orders")

In [0]:
# task 4
orders_once = bronze.dropDuplicates(["order_id"])
exploded = orders_once.withColumn("item", explode(col("items")))

print(bronze.count(), orders_once.count(), exploded.count())

In [0]:
# task 5
online_lines = (exploded
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("order_ts", to_timestamp(col("order_ts")))
    .withColumn("order_date", to_date(col("order_ts")))
    .withColumn("channel", coalesce(col("channel"), lit("online")))
    .withColumn("product", col("item.product"))
    .withColumn("qty", col("item.qty").cast("int"))
    .withColumn("price_eur", col("item.price_eur").cast("double"))
    .withColumn("line_amount_eur", spark_round(col("qty") * col("price_eur"), 2))
    .withColumn("source_file_name", element_at(split(col("source_file"), "/"), -1))
    .drop("items", "item", "_rescued_data", "source_file", "ingested_at"))

display(online_lines.select("city", "channel", "source_file_name").distinct())

In [0]:
# task 6
shop_lines = (spark.table("workspace.coffee.orders")
    .withColumnRenamed("amount_eur", "line_amount_eur")
    .withColumn("order_ts", col("order_date").cast("timestamp"))
    .withColumn("channel", lit("shop"))
    .withColumn("qty", lit(1))
    .withColumn("price_eur", col("line_amount_eur"))
    .withColumn("source_file_name", lit("orders")))

all_lines = online_lines.unionByName(shop_lines)

print(shop_lines.count(), all_lines.count())

In [0]:
# task 7: the rule
good = all_lines.filter(col("customer_id").isNotNull())
rejected = all_lines.filter(col("customer_id").isNull())

In [0]:
# task 7: the checks
keys = good.select("order_id", "product")
assert keys.count() == keys.distinct().count(), "duplicate order lines in silver"
assert good.filter(col("line_amount_eur") <= 0).count() == 0, "zero or negative amounts"
assert good.count() + rejected.count() == all_lines.count(), "rows lost or added by the rule"

In [0]:
# task 7: write
good.write.mode("overwrite").saveAsTable("workspace.coffee.silver_order_lines")
rejected.write.mode("overwrite").saveAsTable("workspace.coffee.silver_rejected_lines")

In [0]:
%sql
-- task 7: check
SELECT COUNT(*) AS lines, ROUND(SUM(line_amount_eur), 2) AS revenue_eur
FROM workspace.coffee.silver_order_lines;

SELECT order_id, product, line_amount_eur, source_file_name
FROM workspace.coffee.silver_rejected_lines;

In [0]:
# task 9
display(spark.table("workspace.coffee.silver_order_lines")
        .select("qty", "line_amount_eur")
        .summary())

In [0]:
%sql
-- task 9
SELECT product FROM workspace.coffee.orders
UNION ALL
SELECT product FROM workspace.coffee.products;

SELECT product FROM workspace.coffee.orders
UNION
SELECT product FROM workspace.coffee.products;

In [0]:
dbutils.widgets.text("fail_on_purpose", "no")

if dbutils.widgets.get("fail_on_purpose") == "yes":
    raise Exception("Silver failed on purpose, to practise a repair")

In [0]:
dbutils.jobs.taskValues.set(key="rejected_lines", value=rejected.count())